# Smart Summarizer: Article Summarization using BERT & T5

This notebook combines all functionality into a single interactive environment:
1. **Text Extraction**: Upload PDF or text files, or paste text directly
2. **Model Loading**: BERT (bert-base-uncased) and T5 (T5-small) models
3. **Summarization**: Generate extractive summaries with BERT and abstractive summaries with T5
4. **Ensemble**: Combine extractive + abstractive outputs
5. **Evaluation**: ROUGE-1, ROUGE-2, ROUGE-L metrics
6. **Batch Pipeline**: Process CSV datasets
7. **Export**: Save results to CSV/Excel

## 1. Imports

In [4]:
! pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 1.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 5.3 MB/s  0:00:00 eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import os
import sys
import time
import tempfile
import re
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import BertModel, BertTokenizer
from transformers import T5ForConditionalGeneration, T5Tokenizer
from rouge import Rouge
from IPython.display import display, HTML, Markdown

import ipywidgets as widgets 
from IPython.display import clear_output

## 2. Text Extraction Utilities

In [7]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from a PDF file."""
    try:
        import PyPDF2
        text = ""
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text.strip()
    except ImportError:
        raise ImportError("PyPDF2 is required for PDF extraction. Install with: pip install PyPDF2")
    except Exception as e:
        raise RuntimeError(f"Failed to extract text from PDF: {e}")


def extract_text_from_txt(txt_path: str) -> str:
    """Extract text from a plain text file."""
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.read().strip()
    except UnicodeDecodeError:
        with open(txt_path, "r", encoding="latin-1") as f:
            return f.read().strip()
    except Exception as e:
        raise RuntimeError(f"Failed to read text file: {e}")


def extract_text(file_path: str) -> str:
    """Extract text from a file based on its extension."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return extract_text_from_pdf(file_path)
    elif ext == ".txt":
        return extract_text_from_txt(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Only .pdf and .txt are supported.")

## 3. Model Loading

In [8]:
def load_models(device):
    """Load BERT and T5 models and tokenizers."""
    print("Loading BERT model (bert-base-uncased)...")
    bert_model_name = "bert-base-uncased"
    bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
    bert_model = BertModel.from_pretrained(bert_model_name).to(device).eval()

    print("Loading T5 model (t5-small)...")
    t5_model_name = "t5-small"
    t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
    t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(device).eval()

    return bert_model, bert_tokenizer, t5_model, t5_tokenizer


# Initialize device and load models
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
bert_model, bert_tokenizer, t5_model, t5_tokenizer = load_models(device)

Using device: cpu
Loading BERT model (bert-base-uncased)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3630.97it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading T5 model (t5-small)...


Loading weights: 100%|██████████| 131/131 [00:08<00:00, 15.44it/s]


## 4. Summary Generation Functions

In [9]:
def split_into_sentences(text: str):
    """Split text into sentences using punctuation boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences


def compute_sentence_embeddings(sentences, model, tokenizer, device="cpu"):
    """Encode sentences with BERT and return normalized embeddings."""
    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoded)
        hidden_states = outputs.last_hidden_state
        attention_mask = encoded.attention_mask.unsqueeze(-1)
        masked_hidden = hidden_states * attention_mask
        sentence_embeddings = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1e-9)
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    return sentence_embeddings.cpu()


def generate_extractive_summary(text: str, model, tokenizer, max_sentences: int = 5, device: str = "cpu") -> str:
    """Select top sentences from the article with BERT as an extractive summary."""
    sentences = split_into_sentences(text)
    if not sentences:
        return ""

    sentence_embeddings = compute_sentence_embeddings(sentences, model, tokenizer, device=device)
    doc_embedding = F.normalize(sentence_embeddings.mean(dim=0, keepdim=True), p=2, dim=1)
    scores = torch.matmul(sentence_embeddings, doc_embedding.T).squeeze(-1)
    top_indices = scores.argsort(descending=True)[: min(max_sentences, len(sentences))]
    top_indices = sorted(top_indices.tolist())
    return " ".join(sentences[i] for i in top_indices)


def generate_summary(text: str, model, tokenizer, prefix: str = "", max_length: int = 130, device: str = "cpu") -> str:
    """Generate a summary for a single text."""
    input_text = prefix + text
    tokenized = tokenizer(
        [input_text], return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            **tokenized,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=2.0,
        )

    return tokenizer.batch_decode(summary_ids, skip_special_tokens=True)[0]


def generate_summary_batch(texts, model, tokenizer, prefix="", max_length=130, batch_size=8, device="cpu"):
    """Generate summaries for a batch of texts."""
    if prefix:
        texts = [prefix + text for text in texts]

    summaries = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        tokenized = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **tokenized,
                max_length=max_length,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
            )

        batch_summaries = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
        summaries.extend(batch_summaries)

    return summaries


def create_ensemble_summary(extractive_summary, t5_summary):
    """Create ensemble summary by combining extractive and abstractive outputs."""
    return f"{extractive_summary} {t5_summary}"


## 5. ROUGE Evaluation

In [10]:
def evaluate_rouge(generated, references):
    """Calculate ROUGE scores."""
    rouge = Rouge()
    try:
        scores = rouge.get_scores(generated, references, avg=True)
        return {
            "rouge-1": scores["rouge-1"]["f"],
            "rouge-2": scores["rouge-2"]["f"],
            "rouge-l": scores["rouge-l"]["f"],
        }
    except Exception as e:
        print(f"ROUGE evaluation error: {e}")
        return {"rouge-1": 0.0, "rouge-2": 0.0, "rouge-l": 0.0}

## 6. Interactive Single Article Summarizer

Use this section to summarize a single article. You can:
- Upload a PDF or text file
- Paste text directly into the text area
- Adjust the max summary length

In [11]:
# Configuration widgets
max_length_widget = widgets.IntSlider(value=130, min=50, max=300, step=10, description="Max Length:")
text_input = widgets.Textarea(placeholder="Paste your article text here...", layout=widgets.Layout(width="100%", height="200px"))
file_upload = widgets.FileUpload(accept=".pdf,.txt", multiple=False, description="Upload File:")
generate_button = widgets.Button(description="✨ Generate Summary", button_style="primary", layout=widgets.Layout(width="200px"))
output_area = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<h3>📄 Input Article</h3>"),
    file_upload,
    widgets.HTML("<p><i>Or paste text below:</i></p>"),
    text_input,
    max_length_widget,
    generate_button,
    output_area
]))

def on_generate_clicked(b):
    with output_area:
        clear_output()
        article_text = ""

        # Get text from uploaded file or pasted text
        if file_upload.value:
            uploaded = list(file_upload.value.values())[0]
            ext = os.path.splitext(list(file_upload.value.keys())[0])[1].lower()
            with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
                tmp.write(uploaded["content"])
                tmp_path = tmp.name
            try:
                article_text = extract_text(tmp_path)
            except Exception as e:
                print(f"Error extracting text: {e}")
                return
            finally:
                os.unlink(tmp_path)
        elif text_input.value.strip():
            article_text = text_input.value.strip()
        else:
            print("Please upload a file or paste text to summarize.")
            return

        if not article_text:
            print("No text found in the uploaded file.")
            return

        # Display extracted text preview
        print(f"📖 Extracted Text Preview ({len(article_text)} characters)")
        print("-" * 60)
        print(article_text[:500] + ("..." if len(article_text) > 500 else ""))
        print()

        # Generate summaries
        print("Generating summaries with BERT extractive + T5 abstractive...")
        start_time = time.time()

        bert_summary = generate_extractive_summary(
            article_text, bert_model, bert_tokenizer, device=device
        )

        t5_summary = generate_summary(
            bert_summary, t5_model, t5_tokenizer, prefix="summarize: ", device=device, max_length=max_length_widget.value
        )

        ensemble_summary = create_ensemble_summary(bert_summary, t5_summary)
        elapsed = time.time() - start_time

        # Display results
        print("\n" + "=" * 60)
        print("🎯 GENERATED SUMMARIES")
        print("=" * 60)

        print("\n🧠 BERT Extractive Summary:")
        print("-" * 40)
        print(bert_summary)

        print("\n🤖 T5 Abstractive Summary:")
        print("-" * 40)
        print(t5_summary)

        print("\n🎯 Ensemble Summary (Combined):")
        print("-" * 40)
        print(ensemble_summary)

        # Stats
        print("\n" + "=" * 60)
        print("📈 Statistics")
        print("=" * 60)
        print(f"Processing Time: {elapsed:.1f}s")
        print(f"Original Length: {len(article_text)} chars")
        print(f"Summary Length: {len(ensemble_summary)} chars")
        compression = (1 - len(ensemble_summary) / len(article_text)) * 100 if len(article_text) > 0 else 0
        print(f"Compression Ratio: {compression:.1f}%")

        # Save results
        results_df = pd.DataFrame({
            "Model": ["BERT Extractive", "T5 Abstractive", "Ensemble"],
            "Summary": [bert_summary, t5_summary, ensemble_summary],
        })

        os.makedirs("output", exist_ok=True)
        csv_output = os.path.join("output", "single_summary_results.csv")
        results_df.to_csv(csv_output, index=False)
        print(f"\nResults saved to {csv_output}")

generate_button.on_click(on_generate_clicked)

## 7. Batch Pipeline (CSV Dataset)

Use this section to process multiple articles from a CSV dataset (like CNN/DailyMail).

In [ ]:
def load_dataset(csv_path, sample_size=10):
    """Load articles from CSV file and sample records."""
    print(f"Loading dataset from {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"Total articles: {len(df)}")

    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"Using sample of {sample_size} articles")

    return df


def run_pipeline(csv_path, sample_size, output_dir):
    """Run the full summarization pipeline."""
    os.makedirs(output_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load data
    df = load_dataset(csv_path, sample_size)

    # Generate summaries
    print("\nGenerating summaries...")
    articles = df["article"].tolist()
    references = df["highlights"].tolist()

    bert_summaries = [
        generate_extractive_summary(article, bert_model, bert_tokenizer, device=device)
        for article in tqdm(articles, desc="Extractive summaries")
    ]

    t5_summaries = generate_summary_batch(
        bert_summaries, t5_model, t5_tokenizer, prefix="summarize: ", device=device, batch_size=4
    )

    ensemble_summaries = [
        create_ensemble_summary(b, t) for b, t in zip(bert_summaries, t5_summaries)
    ]

    # Evaluate
    print("\nEvaluating with ROUGE metrics...")
    rouge_scores = evaluate_rouge(ensemble_summaries, references)

    print(f"\nROUGE-1: {rouge_scores['rouge-1']:.4f}")
    print(f"ROUGE-2: {rouge_scores['rouge-2']:.4f}")
    print(f"ROUGE-L: {rouge_scores['rouge-l']:.4f}")

    # Save results
    results_df = pd.DataFrame({
        "article": articles,
        "reference_summary": references,
        "bert_extractive_summary": bert_summaries,
        "t5_summary": t5_summaries,
        "ensemble_summary": ensemble_summaries,
    })

    csv_output = os.path.join(output_dir, "summarization_results.csv")
    results_df.to_csv(csv_output, index=False)
    print(f"\nResults saved to {csv_output}")

    try:
        excel_output = os.path.join(output_dir, "summarization_results.xlsx")
        results_df.to_excel(excel_output, index=False)
        print(f"Excel output saved to {excel_output}")
    except Exception as e:
        print(f"Could not save Excel (install openpyxl): {e}")

    # Print sample results
    print("\n" + "=" * 60)
    print("SAMPLE RESULTS")
    print("=" * 60)
    for i in range(min(3, len(results_df))):
        print(f"\n--- Article {i+1} ---")
        print(f"Original (truncated): {articles[i][:200]}...")
        print(f"Reference: {references[i]}")
        print(f"Ensemble: {ensemble_summaries[i]}")
        print()

    return rouge_scores


# Run the batch pipeline
SAMPLE_SIZE = 10
OUTPUT_DIR = "output"
DATA_PATH = "train_dataset.csv"

start_time = time.time()
scores = run_pipeline(DATA_PATH, SAMPLE_SIZE, OUTPUT_DIR)
elapsed = time.time() - start_time

print(f"\nPipeline completed in {elapsed:.1f} seconds")

## 8. Results Display

View and analyze the results from the batch pipeline.

In [ ]:
# Display ROUGE scores
if 'scores' in locals():
    print("ROUGE Scores:")
    for metric, value in scores.items():
        print(f"  {metric}: {value:.4f}")

# Load and display results
results_path = os.path.join(OUTPUT_DIR, "summarization_results.csv")
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    display(results_df.head())